In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_from_disk

In [4]:
ds = load_from_disk("data/tokenized")

In [5]:
class TranslationDataset(Dataset):
    def __init__(self, split):
        self.split = split

    def __len__(self):
        return len(self.split)

    def __getitem__(self, i):
        ex = self.split[i]
        return ex["src"], ex["tgt"]

In [9]:
PAD_ID = 0

def collate_fn(batch):
    src = [torch.tensor(s) for s, t in batch]
    tgt_in = [torch.tensor(t[:-1]) for s, t in batch]
    tgt_out = [torch.tensor(t[1:]) for s, t in batch]

    src = pad_sequence(src, batch_first=True, padding_value=PAD_ID)
    tgt_in = pad_sequence(tgt_in, batch_first=True, padding_value=PAD_ID)
    tgt_out = pad_sequence(tgt_out, batch_first=True, padding_value=PAD_ID)
    return src, tgt_in, tgt_out

In [10]:
train_ds = TranslationDataset(ds["train"])
loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)

src, tgt_in, tgt_out = next(iter(loader))
print(src.shape, tgt_in.shape, tgt_out.shape)

torch.Size([64, 23]) torch.Size([64, 28]) torch.Size([64, 28])


In [15]:
import sentencepiece as spm

sp = spm.SentencePieceProcessor(model_file="tokenizer/bpe.model")

In [16]:
print((tgt_in[:, 0] == sp.bos_id()).all())

src0 = src[0][src[0] != PAD_ID].tolist()
tin0 = tgt_in[0][tgt_in[0] != PAD_ID].tolist()
tout0 = tgt_out[0][tgt_out[0] != PAD_ID].tolist()

print(sp.decode(src0))
print(sp.decode(tin0))
print(sp.decode(tout0))
print(tin0)
print(tout0)
print(tout0[-1] == sp.eos_id())

tensor(True)
They trust Tom.
Они доверяют Тому.
Они доверяют Тому.
[2, 791, 3082, 1001, 596, 15768]
[791, 3082, 1001, 596, 15768, 3]
True
